# Bundesliga predictor

In [3]:
import pandas as pd 
import requests
import numpy as np
from collections import defaultdict
from typing import Dict

from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

In [4]:
def get_table(season: int) -> pd.DataFrame:

    url = f"https://api.openligadb.de/getbltable/bl1/{season}"
    data = requests.get(url).json()

    data

    rows = []
    for team in data:
        rows.append({
            "team_name" : team["teamName"],
            "points" : team["points"],
            "win" : team["won"],
            "draw" : team["draw"],
            "loss" : team["lost"],
            "goals_for" : team["goals"],
            "goals_against" : team["opponentGoals"],
            "goal_diff" : team["goalDiff"]
            })
    table = pd.DataFrame(rows)
    table["position"] = table.index + 1
    return table

In [89]:
get_table(2025)

,team_name,points,win,draw,loss,goals_for,goals_against,goal_diff,position
0,FC Bayern München,63,20,3,1,88,23,65,1
1,Borussia Dortmund,52,15,7,2,51,25,26,2
2,TSG Hoffenheim,46,14,4,6,49,31,18,3
3,VfB Stuttgart,46,14,4,6,48,32,16,4
4,RB Leipzig,44,13,5,6,46,33,13,5
5,Bayer 04 Leverkusen,40,12,4,7,44,29,15,6
6,Eintracht Frankfurt,34,9,7,8,48,49,-1,7
7,SC Freiburg,33,9,6,9,34,39,-5,8
8,FC Augsburg,31,9,4,11,30,41,-11,9
9,1. FC Union Berlin,28,7,7,10,29,38,-9,10


The API provided by OpenLigaDB is incorrect for Bundesliga seasons prior to 2023/24, so I had to switch. I found a better API than the previous one, allowing us to implement more stats in our final table!! 

In [13]:
def get_BL_matches(season: int) -> pd.DataFrame :
    url = f"https://raw.githubusercontent.com/openfootball/football.json/master/{season}-{season%100 + 1}/de.1.json"
    data = requests.get(url).json()

    rows = []
    for match in data["matches"]:
        rows.append({
            "matchday": match["round"],
            "date": match["date"],
            "home": match["team1"],
            "away": match["team2"],
            "score_home": match["score"]["ft"][0],
            "score_away": match["score"]["ft"][1]
        })
    matches = pd.DataFrame(rows)
    return matches

In [14]:
def create_table(season: int) -> pd.DataFrame:

    season_matches = get_BL_matches(season)

    teams: Dict[str, Dict[str, int]] = defaultdict(lambda: {
            "wins": 0,
            "draws": 0,
            "losses": 0,
            "goals_for": 0,
            "goals_against": 0,
            "home_wins": 0,
            "home_losses": 0
        })

    for _, row in season_matches.iterrows():
        home = row["home"]
        away = row["away"]

        home_goals = row["score_home"]
        away_goals = row["score_away"]

        teams[home]["goals_for"] += home_goals
        teams[home]["goals_against"] += away_goals
        teams[away]["goals_for"] += away_goals
        teams[away]["goals_against"] += home_goals

        if home_goals > away_goals:
            teams[home]["wins"] += 1
            # teams[home]["home_wins"] += 1
            teams[away]["losses"] += 1
        elif home_goals < away_goals:
            teams[away]["wins"] += 1
            # teams[home]["home_losses"] += 1
            teams[home]["losses"] += 1
        else:
            teams[away]["draws"] += 1
            teams[home]["draws"] += 1

    data = []

    for team, statistics in teams.items():
        goal_diff = statistics["goals_for"] - statistics["goals_against"]
        points = statistics["wins"] * 3 + statistics["draws"] 

        data.append({
            "team_name" : team,
            "points" : points,
            "win" : statistics["wins"],
            "draw" : statistics["draws"],
            "loss" : statistics["losses"],
            "goals_for" : statistics["goals_for"],
            "goals_against" : statistics["goals_against"],
            "goal_diff" : goal_diff,
            # "home_wins" : statistics["home_wins"],
            # "home_losses" : statistics["home_losses"]
        })

    table = pd.DataFrame(data)

    table = table.sort_values(["points", "goal_diff", "goals_for"], ascending= [False, False, False]).reset_index(drop= True)

    table["position"] = table.index + 1

    return table

In [15]:
create_table(2022)

,team_name,points,win,draw,loss,goals_for,goals_against,goal_diff,position
0,FC Bayern München,71,21,8,5,92,38,54,1
1,Borussia Dortmund,71,22,5,7,83,44,39,2
2,RB Leipzig,66,20,6,8,64,41,23,3
3,1. FC Union Berlin,62,18,8,8,51,38,13,4
4,SC Freiburg,59,17,8,9,51,44,7,5
5,Bayer 04 Leverkusen,50,14,8,12,57,49,8,6
6,Eintracht Frankfurt,50,13,11,10,58,52,6,7
7,VfL Wolfsburg,49,13,10,11,57,48,9,8
8,1. FSV Mainz 05,46,12,10,12,54,55,-1,9
9,Borussia Mönchengladbach,43,11,10,13,52,55,-3,10


In [17]:
def prep_data(start_date : int, end_date : int): #end date should be 2024
    season_tables : Dict[str, pd.DataFrame] = {}
    for year in range(start_date, end_date + 1):
        table = get_table(year)
        season_tables[f"{year}"] = table
    feature_rows = []
    target_rows = []
    for i in range(start_date, end_date):
        prev_table = season_tables[f"{i}"].copy().set_index("team_name")
        curr_table = season_tables[f"{i +1}"].copy().set_index("team_name")
        relegated = prev_table.tail(3)
        promoted_stats = relegated.mean().to_dict()

        for team, row in curr_table.iterrows():
            if team in prev_table.index:
                features = prev_table.loc[team][["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]].to_dict()
            else:
                features = {k : promoted_stats[k] for k in ["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]}
            feature_rows.append(features)
            target_rows.append(row["position"])
    X_train = pd.DataFrame(feature_rows)
    y_train = pd.Series(target_rows)

    last_feature_rows = []
    last_table = season_tables[f"{end_date}"].copy().set_index("team_name")
    last_relegated = last_table.tail(2)
    last_promoted_stats = last_relegated.mean().to_dict()
    last_teams = last_table.index.tolist()

    last_promoted_teams = ["FC Köln", "Hamburger SV"]
    last_relegated_teams = last_relegated.index.tolist()

    for team in last_teams:
        if team in last_relegated_teams:
            continue
        features = last_table.loc[team][["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]].to_dict()
        last_feature_rows.append((team, features))
    for team in last_promoted_teams:
        if team not in last_teams:
            feats = {k : last_promoted_stats[k] for k in ["points", "win", "draw", "loss", "goals_for", "goals_against", "goal_diff"]}
            last_feature_rows.append((team, feats))
    latest_features_df = pd.DataFrame([feats for _, feats in last_feature_rows],
                                      index=[t for t, _ in last_feature_rows])
    return X_train, y_train, latest_features_df

Now, we start the random forest using Sklearn.

In [26]:
def train_model(X, y) -> Pipeline: 
    model = Pipeline([
        ("scaler", StandardScaler()),
        ("rf", RandomForestClassifier(
            n_estimators=100,
            max_depth=8,
            random_state=42,
            class_weight="balanced"
        ))
    ])
    model.fit(X, y)
    return model

In [27]:
X, y, df = prep_data(2010, 2024)

In [28]:
model = train_model(X, y)

In [30]:
prob = model.predict_proba(df)
classes = model.named_steps["rf"].classes_
exp_positions = prob.dot(classes)
prediction_df = pd.DataFrame({
        "team": df.index,
        "expected_position": exp_positions
        })

In [31]:
prediction_df.sort_values(["expected_position"], ascending=True)
prediction_df["position"] = prediction_df.index +1

In [32]:
prediction_df

,team,expected_position,position
0,FC Bayern München,1.230000,1
1,Bayer 04 Leverkusen,5.265465,2
2,Eintracht Frankfurt,6.425053,3
3,Borussia Dortmund,6.330141,4
4,SC Freiburg,7.578393,5
5,1. FSV Mainz 05,7.760264,6
6,RB Leipzig,8.913949,7
7,SV Werder Bremen,10.011738,8
8,VfB Stuttgart,8.668466,9
9,Borussia Mönchengladbach,10.093202,10
